
# 00 — Harmonize VAE and univariate discovery targets

This notebook creates one auditable target manifest for the entire AoU replication.

**Inputs**

- Existing VAE protein and chemistry target files in the parent workspace.
- The two supplied OLS/spline-only target files in `inputs/`.
- `config/chemistry_feature_spec.tsv`, which harmonizes chemistry names.

**Key rules**

1. VAE and univariate targets use the same clinical-domain mapping.
2. OLS direction comes from the signed discovery beta.
3. Omnibus spline hits are retained, but their direction is explicitly not evaluable.
4. Longevity/healthspan PRS targets are retained in the manifest but not sent to a disease case-control model.
5. Repeated PRS rows that imply the same external test are grouped within discovery model, domain, and feature. They are not silently discarded.

**Primary outputs**

- `data/targets/target_manifest_all.tsv`
- `data/targets/analysis_targets_grouped.tsv`
- `data/targets/analysis_targets_testable.tsv`
- `data/targets/analysis_targets_not_testable.tsv`
- `data/targets/target_overlap_audit.tsv`


In [ ]:
from pathlib import Path
import os
import sys

# Jupyter's file-browser directory and the kernel working directory can differ.
# Locate the project by its marker instead of trusting Path.cwd().
def find_project_root():
    marker_name = ".aou_combined_replication"
    project_name = "Combined_VAE_linear_AoU_replication"

    env_root = os.getenv("AOU_COMBINED_REPLICATION_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / marker_name).exists():
            return root
        raise FileNotFoundError(
            f"AOU_COMBINED_REPLICATION_ROOT={root} lacks {marker_name}"
        )

    here = Path.cwd().resolve()
    ancestors = [here, *here.parents]
    for candidate in ancestors:
        if (candidate / marker_name).exists():
            return candidate

    matches = []
    for parent in ancestors[:5]:
        patterns = [
            f"*/{marker_name}",
            f"{project_name}/{marker_name}",
            f"*/{project_name}/{marker_name}",
        ]
        for pattern in patterns:
            matches.extend(path.parent for path in parent.glob(pattern))

    matches = sorted(set(path.resolve() for path in matches))
    named = [path for path in matches if path.name == project_name]
    matches = named or matches
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError(
        "Could not locate the combined replication project. Set "
        "AOU_COMBINED_REPLICATION_ROOT explicitly. Candidates found: "
        + str(matches)
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import project_config as cfg
from src import aou_replication_utils as aru

print("Kernel working directory:", Path.cwd().resolve())
print("Project root:", cfg.PROJECT_ROOT)
print("Workspace root:", cfg.WORKSPACE_ROOT)


In [ ]:

import pandas as pd
from IPython.display import display

REQUIRE_VAE_INPUTS = True
MIN_VAE_RUNS = 6

chemistry_spec = aru.load_chemistry_feature_spec(
    cfg.CONFIG_DIR / "chemistry_feature_spec.tsv"
)

vae_protein_file = cfg.first_existing(
    cfg.VAE_PROTEIN_INPUT_CANDIDATES,
    required=REQUIRE_VAE_INPUTS,
)
vae_chemistry_file = cfg.first_existing(
    cfg.VAE_CHEMISTRY_INPUT_CANDIDATES,
    required=REQUIRE_VAE_INPUTS,
)

print("Linear protein input:", cfg.LINEAR_PROTEIN_INPUT)
print("Linear chemistry input:", cfg.LINEAR_CHEMISTRY_INPUT)
print("VAE protein input:", vae_protein_file)
print("VAE chemistry input:", vae_chemistry_file)


In [ ]:
# Load and standardize the supplied OLS/spline candidates.
linear_proteins = aru.harmonize_linear_targets(
    cfg.LINEAR_PROTEIN_INPUT,
    omic_type="proteomics",
    chemistry_spec=chemistry_spec,
)
linear_chemistries = aru.harmonize_linear_targets(
    cfg.LINEAR_CHEMISTRY_INPUT,
    omic_type="chemistries",
    chemistry_spec=chemistry_spec,
)

# Fail early if the purported q-significant source tables contain non-significant rows.
for label, table in {
    "linear proteins": linear_proteins,
    "linear chemistries": linear_chemistries,
}.items():
    bad = table[table["discovery_q"].isna() | table["discovery_q"].ge(0.05)]
    if not bad.empty:
        raise ValueError(
            f"{label} contains {len(bad)} row(s) without discovery q < 0.05"
        )

target_tables = [linear_proteins, linear_chemistries]
protein_crosswalk_audit = pd.DataFrame()

if vae_protein_file is not None:
    vae_proteins = aru.harmonize_vae_targets(
        vae_protein_file,
        omic_type="proteomics",
        chemistry_spec=chemistry_spec,
        min_n_runs=MIN_VAE_RUNS,
    )

    # Legacy VAE targets may have a gene symbol but no UniProt accession.  Use
    # only unambiguous gene-to-UniProt mappings present in the supplied linear
    # protein candidates so the same protein receives the same feature key.
    vae_proteins, protein_crosswalk_audit = (
        aru.apply_protein_gene_uniprot_crosswalk(
            vae_proteins,
            linear_proteins,
        )
    )
    target_tables.append(vae_proteins)

if vae_chemistry_file is not None:
    target_tables.append(
        aru.harmonize_vae_targets(
            vae_chemistry_file,
            omic_type="chemistries",
            chemistry_spec=chemistry_spec,
            min_n_runs=MIN_VAE_RUNS,
        )
    )

targets = pd.concat(target_tables, ignore_index=True)
targets = targets.drop_duplicates(subset="target_id").reset_index(drop=True)

print(f"Raw harmonized target rows: {len(targets):,}")
if not protein_crosswalk_audit.empty:
    print("Protein gene-to-UniProt crosswalk status:")
    display(protein_crosswalk_audit["crosswalk_status"].value_counts())
display(
    targets.groupby(
        ["discovery_framework", "discovery_model", "omic_type"],
        dropna=False,
    ).size().rename("n").reset_index()
)


In [ ]:

# Group repeated discovery rows that imply the same AoU domain-feature test.
analysis_targets = aru.group_targets_for_analysis(targets)
testable = analysis_targets[analysis_targets["clinically_testable"]].copy()
not_testable = analysis_targets[~analysis_targets["clinically_testable"]].copy()
overlap_audit = aru.build_target_overlap_audit(targets)

# The linear inputs were intended to be non-VAE-only. This audit does not assume
# that claim is true; it shows any overlap after common name/domain normalization.
unexpected_overlap = overlap_audit[
    overlap_audit["in_vae"] & overlap_audit["in_univariate"]
].copy()

print(f"Grouped analysis targets: {len(analysis_targets):,}")
print(f"Clinically testable:       {len(testable):,}")
print(f"Not clinically testable:   {len(not_testable):,}")
print(f"VAE/univariate overlap after harmonization: {len(unexpected_overlap):,}")

if not unexpected_overlap.empty:
    print(
        "NOTE: overlap does not necessarily mean the original exclusion was wrong; "
        "the new key collapses names and disease domains more consistently. Review the audit."
    )
    display(unexpected_overlap.head(30))


In [ ]:
# Write all target products. These are the only target files downstream notebooks use.
targets.to_csv(
    cfg.TARGET_DIR / "target_manifest_all.tsv",
    sep="\t",
    index=False,
)
analysis_targets.to_csv(
    cfg.TARGET_DIR / "analysis_targets_grouped.tsv",
    sep="\t",
    index=False,
)
testable.to_csv(
    cfg.TARGET_DIR / "analysis_targets_testable.tsv",
    sep="\t",
    index=False,
)
not_testable.to_csv(
    cfg.TARGET_DIR / "analysis_targets_not_testable.tsv",
    sep="\t",
    index=False,
)
overlap_audit.to_csv(
    cfg.TARGET_DIR / "target_overlap_audit.tsv",
    sep="\t",
    index=False,
)
protein_crosswalk_audit.to_csv(
    cfg.TARGET_DIR / "protein_gene_uniprot_crosswalk_audit.tsv",
    sep="\t",
    index=False,
)

summary = (
    analysis_targets.groupby(
        [
            "discovery_framework",
            "discovery_model",
            "omic_type",
            "analysis_domain",
            "clinically_testable",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        n_analysis_targets=("analysis_target_id", "nunique"),
        n_discovery_rows=("n_discovery_rows", "sum"),
        n_features=("feature_key", "nunique"),
    )
)
summary.to_csv(
    cfg.TARGET_DIR / "target_summary.tsv",
    sep="\t",
    index=False,
)

print("Saved target files to:", cfg.TARGET_DIR)
display(summary)


In [ ]:

# Final human-readable QC views.
print("Not-testable targets and reasons:")
display(
    not_testable[
        [
            "discovery_model_label",
            "omic_label",
            "analysis_domain_label",
            "feature_display",
            "prs_names",
            "not_testable_reason",
        ]
    ].sort_values(["omic_label", "analysis_domain_label", "feature_display"])
)

print("Testable targets, first 30 rows:")
display(
    testable[
        [
            "analysis_target_id",
            "discovery_model_label",
            "omic_label",
            "analysis_domain_label",
            "feature_display",
            "prs_names",
            "discovery_direction",
            "min_discovery_q",
        ]
    ].head(30)
)
